# Лабораторна 3 — VLM у Colab
Розпізнавання 280 синтетичних зображень та 100 рядків IAM. Використано qwen2.5vl:3b, temperature = 0, GPU T4. Потрібні запущений сервер Ollama, завантажена модель і дані у /content/data. Власні фото виконуються лише локально.


In [ ]:
%pip install ollama pandas


In [ ]:
import os
os.chdir("/content")
from pathlib import Path
import time
import ollama
import pandas as pd

MODEL = "qwen2.5vl:3b"
RESULTS = Path("recognition_results.csv")
PROMPT = ("Перепиши дослівно весь текст із зображення, зберігаючи рядки. "
          "Нічого не виправляй, не перекладай і не додавай пояснень.")

def read_results(path):
    if path.exists():
        return pd.read_csv(path, dtype={"id": str}, keep_default_na=False).to_dict("records")
    return []

def recognize_vlm(data, rows):
    completed = {(r["group"], str(r["id"]), int(r["level"])) for r in rows
                 if r["engine"] == "VLM" and r["model"] == MODEL and r["prep"] == "raw"}
    for r in data.itertuples(index=False):
        key = (r.group, str(r.id), int(r.level))
        if key in completed:
            continue
        if not Path(r.path).is_file():
            raise FileNotFoundError(r.path)
        print(r.group, r.id, r.level, "— розпізнавання…", flush=True)
        t0 = time.perf_counter()
        response = ollama.chat(model=MODEL, options=OPTIONS,
                              messages=[{"role": "user", "content": PROMPT,
                                         "images": [str(r.path)]}])
        seconds = time.perf_counter() - t0
        rows.append({"group": r.group, "id": str(r.id), "level": int(r.level),
                     "engine": "VLM", "model": MODEL, "prep": "raw", "gt": r.gt,
                     "pred": response["message"]["content"], "conf": None, "sec": seconds,
                     "lang": r.lang, "psm": r.psm})
        temporary = RESULTS.with_suffix(".tmp.csv")
        pd.DataFrame(rows).to_csv(temporary, index=False)
        temporary.replace(RESULTS)
        completed.add(key)
        print(f"Збережено, {seconds:.1f} с", flush=True)
    return pd.DataFrame(rows)

# У Colab GPU не вимикаємо.
OPTIONS = {"temperature": 0}
synthetic = pd.read_csv("data/ground_truth.csv", dtype={"id": str}, keep_default_na=False)
synthetic["group"], synthetic["lang"], synthetic["psm"] = "synthetic", "ukr", 7
iam = pd.read_csv("data/iam/ground_truth.csv", dtype={"id": str}, keep_default_na=False)
iam["group"], iam["lang"], iam["psm"], iam["level"] = "iam", "eng", 7, -1
columns = ["group", "id", "level", "path", "gt", "lang", "psm"]
data = pd.concat([synthetic[columns], iam[columns]], ignore_index=True)
assert data.groupby("group").size().to_dict() == {"synthetic": 280, "iam": 100}
results = recognize_vlm(data, read_results(RESULTS))
vlm = results[(results.engine == "VLM") & (results.model == MODEL)
              & (results.prep == "raw") & results.group.isin(["synthetic", "iam"])]
assert vlm.groupby("group").size().to_dict() == {"synthetic": 280, "iam": 100}
assert not vlm.duplicated(["group", "id", "level"]).any()
print("Colab завершено: 380 відповідей VLM")
from google.colab import files
files.download(str(RESULTS))
